In [1]:
import numpy as np
import pandas as pd
import Soil_Carbon_Model as md
import os
import openpyxl
from pandas import *

In [2]:
dt = 1 / 24
nyears = 1000
ndays = nyears * 365
nsim = ndays * 24
nnpp = ndays * 24

#Rainfall from Marked Poisson
rain = md.poisson_process(n = int(nsim), alpha = 5.59/1000, lbd = (0.19 * dt))

tsim = np.linspace(1, nnpp, int(nnpp))

days = np.arange(365)
annual_avg_npp = 485.5

shape = np.sin((np.pi*days)/365)             

daily_npp = annual_avg_npp * (shape / np.sum(shape))
hourly_npp = np.repeat(daily_npp, 24)

npp = np.tile(hourly_npp,nyears)

In [3]:
print("Maximum Value is:", max(npp))
print("Average value is:", np.mean(npp))

Maximum Value is: 2.089367842826529
Length is: 8760000
Average value is: 1.3301369863013695


In [4]:
total_rainfall = rain.sum()                                   # total rainfall in 'm'
print(f"Total rainfall over years = {total_rainfall:.3f} meters")

RPY = (total_rainfall / nyears)*1000                          # average rainfall per year in 'mm'
print(f"Average rainfall per year = {RPY:.3f} mm/year")

PET = 1124.9
AI = PET/RPY
print(AI)

Total rainfall over years = 389.224 meters
Average rainfall per year = 389.224 mm/year
2.890108394727422


In [5]:
# Compute TI_DI and TI_Texture

KA_final = 9.0132
n_final = 1.7444
R_max = 3.8

def compute_TI(AI):
    AI = np.asarray(AI, dtype=float)
    numerator   = AI**n_final
    denominator = KA_final**n_final + AI**n_final
    norm_factor = R_max**n_final / (KA_final**n_final + R_max**n_final)
    TI = (numerator / denominator) / norm_factor

    return np.minimum(TI, 1.0)

# ------------------------------------
# TI FROM SAND (SIGMOID)
# ------------------------------------
sand = 38.0   # sand (%)
s50 = 25.0
k   = 0.08

def TI_from_sand(sand):
    return 1.0 / (1.0 + np.exp(-k * (sand - s50)))

TI_sand = TI_from_sand(sand)
TI_AI = compute_TI(AI)
TI = (TI_sand + TI_AI)/2

print("TI_DI =", TI_AI)
print("TI_tex =", TI_sand)
print("TI Total =", TI)

TI_DI = 0.6662532680899118
TI_tex = 0.7388500060842489
TI Total = 0.7025516370870803


In [6]:
#Hydraulic parameterization for dynamic porosity -------------------------------
alpha1 = 580  #hydraulic conductivity shape parameter
beta1 = 6     #hydraulic conductivity shape parameter
alpha2 = 580  #porosity shape parameter
beta2 = 6     #porosity shape parameter
kstex = 0.2   #m/d,
kstot = 1     #m/d
phitex = 0.45  
phitot = 0.60  

In [7]:
# Simulation for 2150 years at dynamic porosity
sim = md.run_julia(rain, npp, kstex, kstot, phitex, phitot, alpha1, beta1, alpha2, beta2, TI)
sim_year = sim.pivot_table(values=['s', 'phi', 'ksstr', 'P', 'DOC', 'A', 'M', 'B',
                                   'Fa', 'Fpa','Flb','Fl', 'Flm', 'Fpl', 'Fbm','Fld', 'Fal',
                                   'lk','npp_', 'L','Fi', 'Flpom', 'decomp'],
                           index='Years')

print(sim_year)

100%|██████████████████████████████████████████████████████████████████████| 8760000/8760000 [18:18<00:00, 7972.12it/s]


                 A           B         DOC        Fa       Fal       Fbm  \
Years                                                                      
1       911.251187   68.793502   62.714979  0.113158  0.003041  0.340596   
2       654.377036   18.845229   90.778028  0.084783  0.001274  0.093302   
3       631.929919    6.232694  129.613638  0.078320  0.000549  0.030857   
4       671.259529    2.824411  179.453287  0.088507  0.000333  0.013983   
5       625.613556    1.834869  268.436199  0.071246  0.000185  0.009083   
...            ...         ...         ...       ...       ...       ...   
996    1807.124458  143.058230  238.650922  0.199663  0.006353  0.708180   
997    1807.074349  134.552879  255.158705  0.190861  0.006019  0.666074   
998    1807.812216  141.793672  245.045602  0.226568  0.007203  0.701905   
999    1808.473948  152.027876  234.977086  0.216302  0.006943  0.752577   
1000   1808.705091  148.532599  240.083770  0.209878  0.006715  0.735274   

           

In [9]:
sim_year.to_csv(r"C:\Users\rishabh.singh\OneDrive - Texas A&M AgriLife\Desktop\Slope\Alvaro-Fuentes\AF.csv", index= True, header = True)
sim_year.to_csv(r"C:\Users\rishabh.singh\OneDrive - Texas A&M AgriLife\Desktop\Slope\Results\AF.csv", index= True, header = True)